# Learning Apache Spark with PySpark

This notebook walks through the most common Spark DataFrame operations using real-world **Dublin Airbnb data**.
Every section is a single focused concept — run the cells top to bottom.

| File | Description |
|------|-------------|
| `listings.csv` | ~7,000 listings — price, neighbourhood, room type, ratings |
| `calendar.csv.gz` | ~2.5M rows — daily availability and pricing |

**Topics covered:** SparkSession · `select` · `filter` · `withColumn` · `groupBy` + `agg` · `orderBy` · `distinct` · null handling · `join` · window functions · string functions

## 0. Imports

All PySpark operations live in `pyspark.sql.functions` (imported as `F`).  
`Window` is needed for window functions later.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## 1. Initialise a SparkSession

The `SparkSession` is the single entry point to all Spark functionality.
- `local[*]` — run on this machine, using all available CPU cores
- `shuffle.partitions=4` — keeps shuffle small for a laptop-sized dataset

In [3]:
spark = (
    SparkSession.builder
    .appName("SparkTeaching")
    .master("local[*]")
    .config("spark.driver.memory", "1g")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version}  |  Cores: {spark.sparkContext.defaultParallelism}")

/usr/local/lib/python3.11/site-packages/pyspark/bin/load-spark-env.sh: line 68: ps: command not found
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/30 09:23:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.3  |  Cores: 4


## 2. Load the Listings CSV

`spark.read.csv()` returns a **DataFrame** — Spark's core distributed data structure.
- `header=True` — first row is column names
- `inferSchema=False` — keep everything as strings; we cast manually
- `multiLine=True` — needed because description/amenity fields span multiple lines

In [4]:
listings_raw = spark.read.csv(
    "/data/listings.csv",
    header=True,
    inferSchema=False,
    multiLine=True,
    quote='"',
    escape='"',
)
print(f"Rows: {listings_raw.count():,}  |  Columns: {len(listings_raw.columns)}")

[Stage 1:>                                                          (0 + 1) / 1]

Rows: 6,945  |  Columns: 79


## 3. Inspect the Schema — `printSchema()`

Every column arrives as `StringType` because `inferSchema=False`.  
We will cast them to proper types in the next step.

In [5]:
cols_of_interest = [
    "id", "neighbourhood_cleansed", "room_type",
    "price", "number_of_reviews", "review_scores_rating"
]
listings_raw.select(cols_of_interest).printSchema()

root
 |-- id: string (nullable = true)
 |-- neighbourhood_cleansed: string (nullable = true)
 |-- room_type: string (nullable = true)
 |-- price: string (nullable = true)
 |-- number_of_reviews: string (nullable = true)
 |-- review_scores_rating: string (nullable = true)



## 4. Preview Rows — `show()`

`show(n)` prints the first `n` rows as a plain-text table.  
Use `truncate=False` to see full values (default truncates at 20 chars).

In [6]:
listings_raw.select("id", "neighbourhood_cleansed", "room_type", "price", "review_scores_rating").show(5)

+------+----------------------+---------------+-------+--------------------+
|    id|neighbourhood_cleansed|      room_type|  price|review_scores_rating|
+------+----------------------+---------------+-------+--------------------+
| 44077|  Dn Laoghaire-Rath...|   Private room| $86.00|                4.82|
| 85156|  Dn Laoghaire-Rath...|   Private room|   NULL|                4.81|
|159889|           Dublin City|   Private room| $48.00|                4.75|
|162809|  Dn Laoghaire-Rath...|   Private room|$101.00|                4.85|
|237465|           Dublin City|Entire home/apt|   NULL|                4.56|
+------+----------------------+---------------+-------+--------------------+
only showing top 5 rows



## 5. `select()` — Choose and Transform Columns

`select()` is Spark's equivalent of SQL `SELECT`.  
Here we also **cast types** and **clean the price** column (strips `$` and commas with a regex).

In [7]:
listings = listings_raw.select(
    F.col("id").cast("long").alias("listing_id"),
    F.col("neighbourhood_cleansed").alias("neighbourhood"),
    F.col("room_type"),
    F.regexp_replace(F.col("price"), r"[\$,]", "").cast("double").alias("price"),
    F.col("number_of_reviews").cast("int").alias("review_count"),
    F.col("review_scores_rating").cast("double").alias("rating"),
    F.col("accommodates").cast("int"),
)
listings.show(5)

+----------+--------------------+---------------+-----+------------+------+------------+
|listing_id|       neighbourhood|      room_type|price|review_count|rating|accommodates|
+----------+--------------------+---------------+-----+------------+------+------------+
|     44077|Dn Laoghaire-Rath...|   Private room| 86.0|         394|  4.82|           2|
|     85156|Dn Laoghaire-Rath...|   Private room| NULL|         357|  4.81|           2|
|    159889|         Dublin City|   Private room| 48.0|         464|  4.75|           1|
|    162809|Dn Laoghaire-Rath...|   Private room|101.0|         397|  4.85|           6|
|    237465|         Dublin City|Entire home/apt| NULL|         599|  4.56|           2|
+----------+--------------------+---------------+-----+------------+------+------------+
only showing top 5 rows



## 6. `filter()` — String Equality

`filter()` (alias: `where()`) keeps only rows matching a condition.  
Spark DataFrames are **lazy** — no work happens until an action like `show()` or `count()` is called.

In [8]:
entire_homes = listings.filter(F.col("room_type") == "Entire home/apt")
print(f"Entire home/apt listings: {entire_homes.count():,}")
entire_homes.show(5)

Entire home/apt listings: 3,925
+----------+-------------+---------------+-----+------------+------+------------+
|listing_id|neighbourhood|      room_type|price|review_count|rating|accommodates|
+----------+-------------+---------------+-----+------------+------+------------+
|    237465|  Dublin City|Entire home/apt| NULL|         599|  4.56|           2|
|    274969|  Dublin City|Entire home/apt| NULL|          10|   4.7|           4|
|    292757|  Dublin City|Entire home/apt|130.0|         507|  4.94|           9|
|    302554|  Dublin City|Entire home/apt|177.0|          78|  4.68|           5|
|    302634|  Dublin City|Entire home/apt| 87.0|         180|  4.64|           2|
+----------+-------------+---------------+-----+------------+------+------------+
only showing top 5 rows



## 7. `filter()` — Numeric Condition

Filter to listings priced under €100 per night.

In [9]:
budget = listings.filter(F.col("price") < 100)
print(f"Listings under €100/night: {budget.count():,}")
budget.select("listing_id", "neighbourhood", "room_type", "price").show(5)

Listings under €100/night: 1,667
+----------+--------------------+---------------+-----+
|listing_id|       neighbourhood|      room_type|price|
+----------+--------------------+---------------+-----+
|     44077|Dn Laoghaire-Rath...|   Private room| 86.0|
|    159889|         Dublin City|   Private room| 48.0|
|    245903|        South Dublin|   Private room| 30.0|
|    302634|         Dublin City|Entire home/apt| 87.0|
|    587218|         Dublin City|   Private room| 90.0|
+----------+--------------------+---------------+-----+
only showing top 5 rows



## 8. `filter()` — Multiple Conditions

Combine conditions with `&` (AND), `|` (OR), `~` (NOT).  
**Always wrap each condition in parentheses** — Python operator precedence bites here.

In [ ]:
good_value = listings.filter(
    (F.col("price") < 100) &
    (F.col("rating") >= 4.5) &
    F.col("rating").isNotNull()
)
print(f"Good value (< €100 & rated ≥ 4.5): {good_value.count():,}")
good_value.select("listing_id", "neighbourhood", "price", "rating").show(10)

## 9. `withColumn()` — Add or Overwrite a Column

`withColumn(name, expr)` adds a new column (or replaces an existing one).  
`F.when().when().otherwise()` is Spark's if / elif / else.

In [ ]:
listings = listings.withColumn(
    "price_tier",
    F.when(F.col("price") < 80, "Budget")
     .when(F.col("price") < 150, "Mid-range")
     .otherwise("Premium")
)
listings.select("listing_id", "price", "price_tier").show(10)

## 10. `groupBy()` + `count()`

`groupBy()` groups rows by one or more columns.  
`.count()` is the simplest aggregation — it counts rows in each group.

In [10]:
listings.groupBy("room_type").count().orderBy(F.desc("count")).show()

+---------------+-----+
|      room_type|count|
+---------------+-----+
|Entire home/apt| 3925|
|   Private room| 2912|
|    Shared room|   92|
|     Hotel room|   16|
+---------------+-----+



## 11. `groupBy()` + `agg()` — Multiple Aggregations

`.agg()` computes several aggregations in a single pass: `avg`, `sum`, `min`, `max`, `count`.  
`F.round(expr, n)` rounds to `n` decimal places.

In [ ]:
listings.groupBy("neighbourhood").agg(
    F.count("*").alias("listing_count"),
    F.round(F.avg("price"), 2).alias("avg_price"),
    F.round(F.avg("rating"), 2).alias("avg_rating"),
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price"),
).orderBy(F.desc("listing_count")).show(10)

## 12. `orderBy()` — Sort Results

Sort by one or more columns. `F.desc()` = descending, `F.asc()` = ascending (default).  
`orderBy` is a **global sort** — it collects all data to the driver. Use with `limit()` on large data.

In [ ]:
listings.filter(F.col("rating").isNotNull()) \
    .select("listing_id", "neighbourhood", "price", "rating") \
    .orderBy(F.desc("rating"), F.asc("price")) \
    .show(10)

## 13. `distinct()` — Unique Values

`distinct()` removes duplicate rows. Combine with `select()` to get unique values of a single column.

In [ ]:
listings.select("neighbourhood").distinct().orderBy("neighbourhood").show(30)

## 14. Handling Nulls — `dropna()` and `fillna()`

Spark treats nulls explicitly — arithmetic and comparisons on null return null.  
- `dropna(subset=[...])` removes rows where *any* of the specified columns is null  
- `fillna({col: default, ...})` replaces nulls with a given value

In [ ]:
print(f"Rows with null rating : {listings.filter(F.col('rating').isNull()).count():,}")
print(f"Rows with null price  : {listings.filter(F.col('price').isNull()).count():,}")

In [ ]:
# Remove rows where price OR rating is null
listings_clean = listings.dropna(subset=["price", "rating"])
print(f"After dropna: {listings_clean.count():,} rows remaining")

In [ ]:
# Fill nulls with defaults instead of dropping
listings_filled = listings.fillna({"rating": 0.0, "review_count": 0})
listings_filled.select("listing_id", "rating", "review_count").show(5)

## 15. Load the Calendar CSV

The calendar has **one row per listing per day** — ~2.5 million rows.  
The `available` column is `"t"` (available) or `"f"` (booked).

In [ ]:
calendar_raw = spark.read.csv(
    "/data/calendar.csv.gz",
    header=True,
    inferSchema=False,
)
print(f"Rows: {calendar_raw.count():,}  |  Columns: {calendar_raw.columns}")
calendar_raw.show(5)

## 16. Clean the Calendar

Convert `available` (`t`/`f` string) into a numeric flag and parse the price string.

In [ ]:
calendar = calendar_raw.select(
    F.col("listing_id").cast("long"),
    F.col("date").cast("date"),
    F.when(F.col("available") == "t", 1).otherwise(0).alias("is_available"),
    F.regexp_replace(F.col("price"), r"[\$,]", "").cast("double").alias("price"),
)
calendar.show(5)

## 17. `filter()` on the Calendar

`is_available == 0` means the day is **booked** (not available to new guests).  
This is the same `filter()` you already know — it works identically on any DataFrame.

In [ ]:
booked = calendar.filter(F.col("is_available") == 0)
print(f"Total booked days across all listings: {booked.count():,}")
booked.show(5)

## 18. `groupBy()` — Per-Listing Occupancy

Group the 2.5M-row calendar down to one summary row per listing.  
`F.sum(F.when(...))` is a **conditional sum** — counts only booked days.

In [ ]:
occupancy = (
    calendar
    .groupBy("listing_id")
    .agg(
        F.count("*").alias("total_days"),
        F.sum(F.when(F.col("is_available") == 0, 1).otherwise(0)).alias("days_booked"),
        F.round(F.avg(F.when(F.col("is_available") == 0, F.col("price"))), 2).alias("avg_booked_price"),
    )
    .withColumn("occupancy_pct", F.round(F.col("days_booked") / F.col("total_days") * 100, 1))
)
occupancy.orderBy(F.desc("occupancy_pct")).show(10)

## 19. `join()` — Combine Two DataFrames

Joins in Spark mirror SQL joins:
- `how="inner"` — only rows present in **both** DataFrames
- `how="left"` — all rows from the left; matched rows from the right (nulls where no match)
- `how="right"`, `how="outer"` also available

In [ ]:
enriched = listings_clean.join(occupancy, on="listing_id", how="left")
print(f"Joined rows: {enriched.count():,}  |  Columns: {len(enriched.columns)}")
enriched.select("listing_id", "neighbourhood", "price", "rating", "occupancy_pct", "avg_booked_price").show(10)

## 20. `groupBy()` on Joined Data — Neighbourhood Summary

Now that listings carry occupancy stats, we can summarise by neighbourhood and room type in one pass.

In [ ]:
summary = (
    enriched
    .groupBy("neighbourhood", "room_type")
    .agg(
        F.count("*").alias("listings"),
        F.round(F.avg("price"), 2).alias("avg_price"),
        F.round(F.avg("occupancy_pct"), 1).alias("avg_occupancy_pct"),
        F.round(F.avg("rating"), 2).alias("avg_rating"),
    )
    .orderBy(F.desc("listings"))
)
summary.show(15)

## 21. Window Functions — `rank()` Within a Group

Window functions operate **across rows** without collapsing them (unlike `groupBy`).  
`partitionBy` = the group, `orderBy` = the ranking criterion within each group.

In [ ]:
window_spec = Window.partitionBy("neighbourhood").orderBy(F.desc("price"))
ranked = listings_clean.withColumn("price_rank", F.rank().over(window_spec))

# Show the 3 most expensive listings per neighbourhood
ranked.filter(F.col("price_rank") <= 3) \
    .select("neighbourhood", "listing_id", "price", "price_rank") \
    .orderBy("neighbourhood", "price_rank") \
    .show(20)

## 22. String Functions

Spark has many built-in string helpers under `F.*`:  
`upper`, `lower`, `trim`, `length`, `contains`, `startswith`, `like`, `split`, `substring`, `concat`, …

In [ ]:
# Find neighbourhoods whose name contains 'city' (case-insensitive)
listings.filter(F.lower(F.col("neighbourhood")).contains("city")) \
    .select("listing_id", "neighbourhood", "price") \
    .show(10)

## 23. `withColumnRenamed()` — Rename a Column

Cleanly rename a single column without rebuilding the whole DataFrame.  
For renaming many columns at once, chain multiple calls or use `select()` with `.alias()`.

In [ ]:
renamed = listings.withColumnRenamed("review_count", "num_reviews")
print("Column names:", renamed.columns)

## 24. Stop the SparkSession

Always stop the session when done to release the JVM and free memory.

In [ ]:
spark.stop()
print("SparkSession stopped.")